In [78]:
from deepface import DeepFace
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2
import os

In [79]:
input_image = "1.png"  # your input image path
db_path = "./pics/"            # your database folder

results = DeepFace.find(
    img_path=input_image,
    db_path=db_path,
    model_name="VGG-Face",
    detector_backend="opencv",
    enforce_detection=True,
    threshold=0.4
)

# results is a list of DataFrames (one per face detected in input image)
print(f"Faces detected in input: {len(results)}")
for i, df in enumerate(results):
    print(f"\nFace {i+1} — {len(df)} match(es) found:")
    print(df[["identity", "distance", "threshold"]])

26-05-04 23:47:51 - Searching 1.png in 6 length datastore
26-05-04 23:47:51 - find function duration 0.290358304977417 seconds
Faces detected in input: 1

Face 1 — 3 match(es) found:
              identity  distance  threshold
0  ./pics/duddin\3.png  0.295935        0.4
1  ./pics/duddin\2.png  0.308374        0.4
2  ./pics/duddin\4.png  0.361102        0.4


In [ ]:
def extract_name(path):
    """Get person name from subfolder structure: db/Alice/img1.jpg → Alice"""
    parts = path.replace("\\", "/").split("/")
    return parts[-2] if len(parts) >= 2 else os.path.splitext(parts[-1])[0]

def display_results(input_image, results):
    for face_idx, df in enumerate(results):
        if df.empty:
            print(f"Face {face_idx+1}: No matches found.")
            continue

        matches = df.copy()
        total = len(matches)

        # --- top row: input image ---
        fig = plt.figure(figsize=(4 + total * 3, 5))
        fig.suptitle(f"Face {face_idx+1} — {total} match(es) found", 
                     fontsize=13, fontweight="bold", y=1.02)

        # input image on the left
        ax_input = fig.add_subplot(1, total + 1, 1)
        ax_input.imshow(Image.open(input_image))
        ax_input.set_title("Input", fontsize=10, color="steelblue")
        ax_input.axis("off")

        # matched images on the right
        for i, (_, row) in enumerate(matches.iterrows()):
            name = extract_name(row["identity"])
            addr = row["identity"]
            distance = row["distance"]
            threshold = row["threshold"]
            confidence = round((1 - distance / threshold) * 100, 1)

            ax = fig.add_subplot(1, total + 1, i + 2)
            ax.imshow(Image.open(row["identity"]))

            # color border based on confidence
            color = "green" if confidence >= 70 else "orange"
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)

            ax.set_title(f"{name}\ndist: {distance:.3f}\n{confidence}% conf.\n{addr}",
                         fontsize=9, color=color)
            ax.axis("off")

        plt.tight_layout()
        plt.show()

display_results(input_image, results)